In [2]:
from langchain.pydantic_v1 import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_function
class Sentiment(BaseModel):
    """Define o sentimento e o idioma da mensagem enviada"""
    sentimento: str = Field(description="Sentimento do texto. Deve ser 'pos', 'neg' ou 'nt' para não definido")
    lingua: str = Field(description="Lingua que o texto foi escrito (deve estar no charset UTF-8)") 

tool_sentimento = convert_to_openai_function(Sentiment)
tool_sentimento

{'name': 'Sentiment',
 'description': 'Define o sentimento e o idioma da mensagem enviada',
 'parameters': {'type': 'object',
  'properties': {'sentimento': {'description': "Sentimento do texto. Deve ser 'pos', 'neg' ou 'nt' para não definido",
    'type': 'string'},
   'lingua': {'description': 'Lingua que o texto foi escrito (deve estar no charset UTF-8)',
    'type': 'string'}},
  'required': ['sentimento', 'lingua']}}

In [4]:
texto = "Eu gosto muito de pizza de paulista"

In [10]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "Analise o texto e categorize-o conforme as intruções"),
    ("user", "{input}")
])

chat = ChatOpenAI()

chain = prompt | chat.bind(
    functions=[tool_sentimento],
    function_call={"name": "Sentiment"})

In [11]:
chain.invoke({"input": texto})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentimento":"pos","lingua":"pt"}', 'name': 'Sentiment'}}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 134, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-329f444e-d618-44b0-9d10-75a319308400-0')

In [12]:
chain.invoke({"input": "May the Force be with you"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"sentimento":"nt","lingua":"en"}', 'name': 'Sentiment'}}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 131, 'total_tokens': 142, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-3e889166-18dc-47eb-ae23-38d9a7e9e7bf-0')

## Categorização de texto

In [13]:
solicitacoes = [
    "Meu computador está travando toda vez que tento abrir um programa. O que devo fazer?",
    "Não consigo acessar minha conta. A senha não está funcionando!",
    "Estou tendo dificuldades para instalar o software de design gráfico. Pode me ajudar?",
    "Gostaria de saber como atualizar meu endereço de entrega na plataforma.",
    "O meu laptop não está ligando. Acho que é um problema no cabo de energia.",
    "Não consigo ver meus arquivos no aplicativo. Existe alguma solução?"
]

In [17]:
from enum import Enum
from langchain.pydantic_v1 import BaseModel, Field

class SetorEnum(str, Enum):
    suport_hardware = "suport_hardware"
    suport_software = "suport_software"
    suport_conta    = "suport_conta"
    outros          = "outros"


class DirecionaSuporte(BaseModel):
    """Direciona a solicitação de suporte para o setor responsável"""
    setor: SetorEnum

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_function

tool_direcionamento_suporte = convert_to_openai_function(DirecionaSuporte)
tool_direcionamento_suporte

{'name': 'DirecionaSuporte',
 'description': 'Direciona a solicitação de suporte para o setor responsável',
 'parameters': {'type': 'object',
  'properties': {'setor': {'description': 'An enumeration.',
    'enum': ['suport_hardware', 'suport_software', 'suport_conta', 'outros'],
    'type': 'string'}},
  'required': ['setor']}}

In [19]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

# Criar o prompt com as instruções detalhadas para o modelo
system_message = '''Pense com cuidado ao categorizar o texto conforme as instruções.
Questões relacionadas a problemas de hardware, como falha no computador, laptop ou
outros dispositivos físicos devem ser direcionadas para "suporte_hardware".
Questões relacionadas a problemas com software, como instalação, erros ao abrir programas,
etc., devem ser direcionadas para "suporte_software".
Problemas relacionados ao acesso de conta, como recuperação de senha, problemas de login,
devem ser direcionadas para "suporte_conta".
Mensagens que não se encaixam nessas categorias devem ser direcionadas para "outros".
'''

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", "{input}")
])

chain = (prompt | chat.bind(functions=[tool_direcionamento_suporte],
                            function_call={"name": "DirecionaSuporte"})
                            | JsonOutputFunctionsParser())

In [ ]:
solicitacao = solicitacoes[0]
resposta    = chain.invoke({"input": solicitacao})
prompt(f"Solicitação: {solicitacao}")
prompt(f"Resposta: {resposta}")